# Examen Segundo Parcial — Análisis de Datos (CETI IDS)
**Registro:** 23310032  
**Nombre:** Alfredo Alejandro Longoria Alcantar  
**Grupo:** 7B  
**Fecha:** 27/10/2026  

## Caso: Heart Disease — prevalencia de `target = 1` por `sex × cp`
En este caso hago una tabla de contingencia de sexo contra tipo de dolor en el pecho para ver qué porcentaje
de pacientes tiene enfermedad cardiaca en cada combinación, y luego la grafico como heatmap.

Variables: `sex` (0 = mujer, 1 = hombre), `cp` (tipo de dolor torácico) y `target` (1 = tiene enfermedad).

*Nota:* la URL del examen marcaba error 404, así que usé una copia pública del mismo dataset (303 registros, mismas columnas).

In [ ]:
# === Celda 1. Carga y preparación del dataset Heart Disease ===
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

COLS_UCI = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
            "exang","oldpeak","slope","ca","thal","num"]

FUENTES = [
    # URL indicada en el examen (actualmente devuelve error 404)
    ("https://raw.githubusercontent.com/sassoftware/examples/main/data/heart_disease.csv", {}),
    # Copia pública del mismo dataset Heart Disease (303 pacientes, columna target)
    ("https://raw.githubusercontent.com/kb22/Heart-Disease-Prediction/master/dataset.csv", {}),
    # Fuente original UCI (Cleveland), sin encabezado y con '?' como faltante
    ("https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data",
     {"header": None, "names": COLS_UCI, "na_values": "?"}),
    ("heart_disease.csv", {}),   # copia local subida a Colab
]
df = None
for fuente, opciones in FUENTES:
    try:
        df = pd.read_csv(fuente, **opciones)
        print("Datos cargados desde:", fuente)
        break
    except Exception as e:
        print("No se pudo leer", fuente, "->", type(e).__name__)
if df is None:
    raise FileNotFoundError("Sube el CSV a Colab con el nombre heart_disease.csv")

df.columns = df.columns.str.strip().str.lower()

# Algunas versiones del dataset traen 'num' (0-4) en lugar de 'target'
if "target" not in df.columns and "num" in df.columns:
    df["target"] = (df["num"] > 0).astype(int)

# Solo se necesitan sex, cp y target; se eliminan filas sin esas variables
df = df.dropna(subset=["sex", "cp", "target"]).copy()
df["target"] = (pd.to_numeric(df["target"], errors="coerce") > 0).astype(int)

# Etiquetas legibles (codificación estándar UCI/Kaggle)
mapa_sex = {0: "Mujer", 1: "Hombre"}
# Se muestra el código original junto a la etiqueta para que no haya ambigüedad.
if set(df["cp"].unique()) <= {1, 2, 3, 4}:          # versión UCI original (1-4)
    mapa_cp = {1: "1·Angina típica", 2: "2·Angina atípica", 3: "3·No anginoso", 4: "4·Asintomático"}
else:                                                # versión recodificada 0-3 (las frecuencias coinciden
    mapa_cp = {3: "3·Angina típica", 1: "1·Angina atípica",   # con UCI: 0 es el grupo más grande =
               2: "2·No anginoso", 0: "0·Asintomático"}       # asintomático; 3 el más pequeño = típica)

df["sexo"] = df["sex"].map(mapa_sex).fillna(df["sex"].astype(str))
df["dolor"] = pd.Categorical(df["cp"].map(mapa_cp).fillna(df["cp"].astype(str)),
                             categories=list(mapa_cp.values()), ordered=True)

print(df.shape)
print("Prevalencia global de target=1:", round(df["target"].mean() * 100, 1), "%")
df[["age", "sex", "cp", "chol", "thalach", "target"]].head()

In [ ]:
# === Celda 2. Tabla de contingencia sex × cp ===
# n por celda (tamaño de la base de cada prevalencia)
n_tab = pd.crosstab(df["sexo"], df["dolor"], margins=True, margins_name="Total")

# casos con target = 1 por celda
casos_tab = pd.crosstab(df["sexo"], df["dolor"], values=df["target"], aggfunc="sum",
                        margins=True, margins_name="Total").fillna(0).astype(int)

# prevalencia (%) = casos / n  -> equivalente a la media de target en cada celda
prev_tab = (pd.crosstab(df["sexo"], df["dolor"], values=df["target"], aggfunc="mean",
                        margins=True, margins_name="Total") * 100).round(1)

# Intervalo de confianza 95% de Wilson para medir la inestabilidad de cada prevalencia
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    den = 1 + z**2 / n
    centro = (p + z**2 / (2 * n)) / den
    margen = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return (max(0, centro - margen) * 100, min(1, centro + margen) * 100)

filas = []
for s in n_tab.index.drop("Total"):
    for c in n_tab.columns.drop("Total"):
        n, k = int(n_tab.loc[s, c]), int(casos_tab.loc[s, c])
        li, ls = wilson(k, n)
        filas.append({"sexo": s, "cp": c, "n": n, "casos_target1": k,
                      "prevalencia_%": round(k / n * 100, 1) if n else np.nan,
                      "IC95_inf": round(li, 1), "IC95_sup": round(ls, 1),
                      "amplitud_IC": round(ls - li, 1)})
resumen = pd.DataFrame(filas)

UMBRAL_N = 30   # celdas con menos de 30 pacientes se consideran inestables
resumen["estable"] = np.where(resumen["n"] >= UMBRAL_N, "Sí", "NO (n<30)")

print("Pacientes por celda (n):");            display(n_tab)
print("Casos con target = 1:");               display(casos_tab)
print("Prevalencia de target = 1 (%):");      display(prev_tab)
print("Detalle con IC95% de Wilson:")
display(resumen.sort_values("n"))

### ¿Por qué lo hice así?
- Usé `crosstab` con el **promedio** de `target` en vez de solo contar casos, porque al ser 0/1 el promedio ya es
  la prevalencia. Si solo contara enfermos, los hombres saldrían más altos solo porque hay más hombres en los datos.
- Agregué el **n** de cada celda porque un porcentaje sin saber de cuántos pacientes sale puede engañar
  (no es lo mismo 50 % de 4 personas que 50 % de 100).
- Calculé un **intervalo de confianza (Wilson)** para ver qué tan confiable es cada porcentaje. Elegí Wilson y no el
  normal porque funciona mejor cuando hay pocos datos.
- Tomé **n < 30** como límite para decir que una celda tiene muy pocos casos.

In [ ]:
# === Celda 3. Heatmap anotado de prevalencia ===
prev = prev_tab.drop(index="Total", columns="Total")
n_cel = n_tab.drop(index="Total", columns="Total")
k_cel = casos_tab.drop(index="Total", columns="Total")

# Anotación: prevalencia + casos/n; se marca con * la celda inestable
anot = prev.copy().astype(object)
for s in prev.index:
    for c in prev.columns:
        n = n_cel.loc[s, c]
        marca = " *" if n < UMBRAL_N else ""
        anot.loc[s, c] = f"{prev.loc[s, c]:.1f}%{marca}\n({k_cel.loc[s, c]}/{n})"

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(prev, annot=anot, fmt="", cmap="YlOrRd", vmin=0, vmax=100,
            linewidths=1.5, linecolor="white",
            cbar_kws={"label": "Prevalencia de target = 1 (%)"}, ax=ax,
            annot_kws={"fontsize": 11})

# Recuadro punteado sobre celdas con pocos casos
for i, s in enumerate(prev.index):
    for j, c in enumerate(prev.columns):
        if n_cel.loc[s, c] < UMBRAL_N:
            ax.add_patch(plt.Rectangle((j + 0.04, i + 0.04), 0.92, 0.92, fill=False,
                                       edgecolor="black", lw=2, linestyle="--"))

ax.set_title("Prevalencia de enfermedad cardiaca (target = 1) por sexo y tipo de dolor torácico\n"
             f"* y borde punteado: celda con n < {UMBRAL_N} (prevalencia inestable)", fontsize=12)
ax.set_xlabel("Tipo de dolor torácico (cp)")
ax.set_ylabel("Sexo")
plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("Celdas inestables (n <", UMBRAL_N, "):")
display(resumen[resumen["n"] < UMBRAL_N][["sexo", "cp", "n", "casos_target1",
                                         "prevalencia_%", "IC95_inf", "IC95_sup"]])

### Sobre el heatmap
Usé la escala `YlOrRd` (amarillo = baja, rojo = alta) porque la prevalencia va en una sola dirección y el rojo se
entiende fácil como "más riesgo". Fijé la escala de 0 a 100 % para que los colores no exageren diferencias pequeñas.
En cada celda puse el porcentaje y los casos/total, y marqué con `*` y borde punteado las celdas con menos de 30 pacientes.

### ¿Qué celdas tienen muy pocos casos?
Las de **mujeres con angina típica** y **mujeres con angina atípica** son las más pequeñas, y también **hombres con
angina típica**; todas quedan por debajo de 30 pacientes (los números exactos salen en la tabla `resumen`).
En esas celdas un solo paciente más o menos cambia mucho el porcentaje, y el intervalo de confianza es muy ancho.

En el heatmap esto significa que esas celdas pueden verse muy rojas o muy claras solo por casualidad, así que no las
tomo como un resultado firme. Las comparaciones confiables son las celdas grandes (por ejemplo hombres asintomáticos)
y los totales por fila y columna.

### Conclusión
El riesgo de enfermedad cardiaca cambia según la combinación de sexo y tipo de dolor, no solo por una de las dos
variables. Por eso, para un triaje en una unidad de salud conviene usar ambas al mismo tiempo. Los grupos con muchos
pacientes y prevalencia alta deberían tener prioridad para estudios como electrocardiograma y revisión con cardiología.

Las celdas de mujeres tienen muy pocos datos, así que no se puede decir que su riesgo sea bajo. Recomendaría no
descartar enfermedad cardiaca en mujeres solo porque su dolor sea "atípico", juntar más registros de pacientes
mujeres y siempre mostrar el porcentaje junto con el número de pacientes, para que quien tome decisiones sepa qué tan
confiable es cada dato.